# Programming Python for Bioinformatics (WBT-MBT2-25E)

---

## Python Programming Crash Course
### Wojciech Dec
<br>

## Part 3 - Functional Programming and Comprehensions

We will become familiar with the keywords:
* `lambda`, `yield`

built-in functions:
* `zip`, `map`, `filter`, `next`

We will also examine the syntax of comprehensions (for lists, dictionaries, and sets) and discuss generators.

### Generators

A generator is an iterator that uses **lazy evaluation**.
**Instead of storing all elements in memory, it defines a procedure for producing the next element**. A value is computed only when requested, typically by calling the `next()` function. This approach is memory-efficient, because only one element is produced at a time.

Generators are a special type of iterator. In a loop, Python repeatedly calls `next()` until a `StopIteration` exception is raised.

Generators are particularly useful when working with very large datasets or files, where loading all data into memory would be impractical. For example, when processing large sequencing files (such as FASTA or FASTQ), a generator can yield one record at a time, allowing the program to process the file sequentially without storing the entire dataset in memory.

Later we will see different ways to implement generators and identify which built-in functions return generators.

#### Memory usage

The following example compares the memory usage of a lazy iterable (`range`) and a materialized `list` containing the same values.

In [1]:
import sys

generator = range(1000000)
generator_no_longer = list(range(1000000))

# size in bytes

print(sys.getsizeof(generator))
print(sys.getsizeof(generator_no_longer))

48
8000056


The object returned by `range()` does not store all numbers in memory.
Instead, it stores only the information required to generate them (start, stop, step).

In contrast, `list(range(...))` creates and stores every element explicitly, which requires substantially more memory.

#### Defining Generators

Generators can be defined in several ways.
Inside functions, a generator is created using the `yield` statement, which produces values one at a time (instead of returning a single value with return).

In [2]:
def fibonacci():
    
    current, previous = 0, 1
    
    while True:
        
        yield current
        
        current, previous = current + previous, current


fib = fibonacci()

In [3]:
print(type(fibonacci), type(fib))

<class 'function'> <class 'generator'>


In [4]:
for i in range(10):
    print(next(fib), end=" ")

0 1 1 2 3 5 8 13 21 34 

Each call to `next()` resumes the function and produces the next value in the sequence.

#### Generator Expressions

Generators can also be created using a generator expression, which resembles a comprehension but uses parentheses `()` instead of square brackets `[]`.

In [5]:
# This is NOT a "tuple comprehension" — it creates a generator
x = (i + 1 for i in range(2))

print(next(x))
print(next(x))
print(next(x))

1
2


StopIteration: 

**Important:** A generator can be iterated only once.
Once all elements have been produced, the generator is exhausted, and further calls to `next()` raise the `StopIteration` exception.

Generators also do not support indexing, because they do not store their elements in memory.

However, it is possible to send a value into a generator using the `send()` method. This allows external code to provide input to the generator while it is running.

### Lambda — Anonymous Functions

Lambda functions are small anonymous (unnamed) functions defined inline using the `lambda` keyword.

Syntax:
<p style="text-align:center;">lambda [arguments] : [expression]</p>

A lambda function may accept multiple arguments but must contain exactly one expression, whose value becomes the return value.

In [ ]:
(lambda x, y: y + 2 * x)(34, 1)

A lambda function can also be assigned to a variable:

In [6]:
f = lambda x: x % 2 == 0

print(f(13))
print(f(31))

False
False


However, this goes against the main idea of `lambda` functions, which are intended to be anonymous, short-lived functions used inline.

A `lambda` function must contain exactly one expression. However, that expression can itself contain nested conditional expressions, which means lambdas can still become arbitrarily complex.

For example:

In [7]:
x = (lambda x: "1" if x == 1 else ("2" if x == 2 else ("3" if x == 3 else "cheers")))(3)

print(x)

3


Although valid, deeply nested lambda expressions quickly become difficult to read and maintain.
In such cases, defining a regular function with def is usually the better choice.

### Higher-Order Functions: `map` and `filter`

Functions such as `map` and `filter` return lazy iterators (similarly to `zip`, `range`, or file objects returned by `open`).
Because they operate lazily, values are produced only when the iterator is consumed.

These are called higher-order functions because they take another function as an argument (and apply it to elements of an iterable).

#### `map`

`map(function, iterable)` applies a function to each element.

In [8]:
print(map(lambda x: x % 2 == 0, range(10)))

To obtain the values, the iterator must be consumed:

In [9]:
list(map(lambda x: x % 2 == 0, range(10)))

[True, False, True, False, True, False, True, False, True, False]

#### `filter`

`filter(function, iterable)` returns only the elements for which the function evaluates to `True`.

In [10]:
list(filter(lambda x: x % 2 == 0, range(10)))

[0, 2, 4, 6, 8]

Conceptually, filter is equivalent to the generator expression:

`(item for item in iterable if function(item))`


The function passed to `map` or `filter` does not need to be a `lambda`:

In [11]:
def my_function(x):
    return x ** x

list(map(my_function, range(5)))

[1, 1, 4, 27, 256]

#### Potential Pitfalls

##### Lazy Evaluation and Mutability

Lazy iterators are efficient, but they can sometimes lead to unintuitive behaviour. Two common issues are:

* iterator exhaustion

* dependence on mutable data

In [12]:
# A lazy iterator can be consumed only once.

def fun_1(x): 
    return x + 1

def fun_2(x):
    return x + 2


y = map(fun_1, range(10))
print("y =", list(y))

# The iterator has already been consumed by list()
print(list(map(fun_2, y)))

y = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[]


Because lazy iterators compute values only when consumed, their result may depend on changes made to the original object after the iterator is created.

In [13]:
# Mutability of the source object

a = [1, 2, 3, 4]

# Values will be computed later
res = filter(lambda x: x % 2 == 0, a)

a.append(11)
a.append(12)

print(list(res))

[2, 4, 12]


### Comprehensions

Comprehensions are a Pythonic and often efficient way to construct collections.
They are generally preferred over `map()` / `filter()` with `lambda` for readability.

However, they are **eagerly evaluated** — all elements are computed immediately and **stored in memory**.

Syntax:
`list = [expression for item in iterable]`

#### Creating Lists

A `list` can be created using a standard `for` loop:

In [14]:
new_numbers = []

for x in range(10):
    new_numbers.append(x + 1)

print(new_numbers)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


The same result can be obtained more concisely using a **list comprehension**:

In [15]:
new_numbers = [x + 1 for x in range(10)]

print(new_numbers)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


##### List with a Conditional (`if`)

In [16]:
some_numbers_list = []

for x in range(20):
    if x % 2 == 0:
        some_numbers_list.append(x)

print(some_numbers_list)

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


Equivalent list comprehension:

In [17]:
some_numbers_list = [x for x in range(20) if x % 2 == 0]

print(some_numbers_list)

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


##### List with Conditional Expression (`if` ... `else`)

In [18]:
some_numbers_list = []

for x in range(10):
    if x % 2 == 0:
        some_numbers_list.append(x)
    else:
        some_numbers_list.append(66)

print(some_numbers_list)

[0, 66, 2, 66, 4, 66, 6, 66, 8, 66]


Equivalent list comprehension:

In [19]:
some_numbers_list = [x if x % 2 == 0 else 66 for x in range(10)]

print(some_numbers_list)

[0, 66, 2, 66, 4, 66, 6, 66, 8, 66]


#### Nested Lists

Creating a list of lists using loops:

In [20]:
some_numbers_lists = []

for i in range(3):
    
    list_i = []
    
    for j in range(3):
        list_i.append(j)
        
    some_numbers_lists.append(list_i)

print(some_numbers_lists)

[[0, 1, 2], [0, 1, 2], [0, 1, 2]]


Equivalent list comprehension:

In [21]:
some_numbers_lists = [[j for j in range(3)] for i in range(3)]

print(some_numbers_lists)

[[0, 1, 2], [0, 1, 2], [0, 1, 2]]


List comprehensions:

* are more concise and readable than explicit loops (for simple cases)

* are typically faster in practice

* should be avoided when the logic becomes too complex or nested, as readability decreases

##### Set and Dictionary Comprehensions
The same comprehension syntax can be used to create **sets** and **dictionaries**.

In [22]:
new_set = {x for x in range(10) if x % 2 != 0}

print(new_set)

{1, 3, 5, 7, 9}


In [23]:
new_dict = {x: x**2 for x in range(11, 20)}

print(new_dict)

{11: 121, 12: 144, 13: 169, 14: 196, 15: 225, 16: 256, 17: 289, 18: 324, 19: 361}


##### The `zip` Function

The `zip()` function returns a lazy iterator of tuples, combining elements from multiple iterables.

In [24]:
a = ['a', 'b', 'c', 'd', 'e', 'f']
b = list(range(6))

for x, y in zip(a, b):
    print(x, y)

a 0
b 1
c 2
d 3
e 4
f 5


##### Creating Dictionaries with `zip`

A common use of `zip()` is to construct dictionaries:

In [25]:
my_dict = dict(zip(a, b))

print(my_dict)

{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5}


Equivalent comprehension:

In [26]:
new_dict = {key: value for key, value in zip(a, b)}

print(new_dict)

{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5}


##### Unequal Lengths

If the input iterables have different lengths, `zip()` stops at the shortest one. Elements without a matching pair are **ignored**.

In [27]:
a = ['a', 'b', 'c', 'd', 'e', 'f']
b = list(range(3))

my_dict = dict(zip(a, b))

print(my_dict)

{'a': 0, 'b': 1, 'c': 2}


### Decorators

A decorator is a function that modifies the behavior of another function without changing its source code.

It works by wrapping the original function inside another function.

In [28]:
import time

def log_execution_time(func):
    
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Function '{func.__name__}' executed in {end_time - start_time:.6f} seconds.")
        return result
    
    return wrapper


@log_execution_time
def gc_content(seq):
    gc_count = seq.count('G') + seq.count('C')
    return (gc_count / len(seq)) * 100


@log_execution_time
def reverse_complement(seq):
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join(complement[base] for base in reversed(seq))

In [29]:
dna_seq = "ATGCTAGCTAGCGTACGATCG"

gc = gc_content(dna_seq)
print(f"GC Content: {gc:.2f}%")

rev_comp = reverse_complement(dna_seq)
print(f"Reverse Complement: {rev_comp}")

Function 'gc_content' executed in 0.000005 seconds.
GC Content: 52.38%
Function 'reverse_complement' executed in 0.000008 seconds.
Reverse Complement: CGATCGTACGCTAGCTAGCAT


Multiple decorators can be applied to a single function.

In [30]:
def polite_wrapper(func):
    def wrapper(*args, **kwargs):
        print("Please.")
        result = func(*args, **kwargs)
        print("Thank you.")
        return result
    return wrapper

@polite_wrapper
@log_execution_time
def calc_sth():
    x = 0
    for i in range(1, 1000):
        for j in range(1, 1000):
            x += i / j
    print(x)


calc_sth()

Please.
3738493.194844967
Function 'calc_sth' executed in 0.036563 seconds.
Thank you.


##### Order of Application

Decorators are applied from bottom to top:

`@kulturalnie`

`@log_execution_time`

`def calc_sth():`

is equivalent to:

calc_sth = kulturalnie(log_execution_time(calc_sth))

In Jupyter notebooks, "magic commands" provide convenient tools for tasks such as timing code execution. `%%timeit` must be placed at the very beginning of a cell.

In [31]:
%%timeit

def calc_sth():
    x = 0
    for i in range(1, 1000):
        for j in range(1, 1000):
            x += i / j
    #print(x)


calc_sth()

36.6 ms ± 1.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Random

The random module provides functions for generating pseudo-random numbers from various distributions.
It uses the **Mersenne Twister algorithm**.

In [32]:
import random

# The seed determines the internal state of the generator and can be used to ensure reproducibility
random.seed(7122023)

# random integer from the closed interval [a, b]
final_grade = random.randint(0, 5)

students = ["John", "Tom", "Julia", "Eve"]

# random element from a sequence
student = random.choice(students)

print(f"{student} will recive {final_grade}!")

John will recive 5!


In [33]:
import string

# Random Sampling with Replacement
text = " ".join([
    "".join(random.choices(string.ascii_lowercase, k=random.randint(1, 10)))
    for _ in range(random.randint(10, 20))
])

print("Exercise 1. " + text.capitalize() + ".")

Exercise 1. Iqqrpvp vmh ujzxvjd pcl oo iauvarqpq yd leeu zk mp wins hqr rlxp eyfbo hxezudnyzy.


In [34]:
random.sample(students, k=2) # Returns k unique elements (no repetition).

['Eve', 'Tom']

In [35]:
# Random from a Range

# equivalent to choice(range(...))
score = random.randrange(0, 100, 5)
print(score)

55


In [36]:
# Continuous Distributions

# from [0, 1)
print(random.random())

# from [a, b]
print(random.uniform(10, 20))

0.7258058265902769
14.860756019580869


In [37]:
# Shuffling

print(students)
random.shuffle(students) # shuffles the list in place
print(students)

['John', 'Tom', 'Julia', 'Eve']
['Eve', 'John', 'Julia', 'Tom']


**Exercise 1**. Transform the data stored in `table_data` into a dictionary of the form:

`{"UUU": "F", ...}`

A two-line solution is possible.

In [7]:
table_data = """
UUU F      CUU L      AUU I      GUU V
UUC F      CUC L      AUC I      GUC V
UUA L      CUA L      AUA I      GUA V
UUG L      CUG L      AUG M      GUG V
UCU S      CCU P      ACU T      GCU A
UCC S      CCC P      ACC T      GCC A
UCA S      CCA P      ACA T      GCA A
UCG S      CCG P      ACG T      GCG A
UAU Y      CAU H      AAU N      GAU D
UAC Y      CAC H      AAC N      GAC D
UAA Stop   CAA Q      AAA K      GAA E
UAG Stop   CAG Q      AAG K      GAG E
UGU C      CGU R      AGU S      GGU G
UGC C      CGC R      AGC S      GGC G
UGA Stop   CGA R      AGA R      GGA G
UGG W      CGG R      AGG R      GGG G
"""

**Exercise 2**. Use the `filter` function to keep only sequences longer than a threshold (4) from the list `reads`.

In [6]:
reads = ["ATG", "ATGCGT", "A", "ATGCTAGC"]

**Exercise 3**. Extract all k-mers from a sequence using a comprehension. A k-mer is a substring of length k, obtained by sliding a window of length k along a sequence. For example, for the sequence `"ATGCG"` and `k = 3`, the k-mers are: `"ATG", "TGC", "GCG"`.


In [10]:
k=3
seq = "ATTTGCAGCTACAAGGTTA"

**Exercise 4**. Use `map` and `lambda` to calculate the GC content for each sequence in the `sequences` list.

In [8]:
sequences = ["ATGC", "GGCC", "ATAT"]

**Exercise 5**. Write a generator that yields random DNA sequences of a given (variable) length.